<a href="https://colab.research.google.com/github/nnott3/KilterTransformer/blob/main/gpt_shuffle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# init

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/nnott3/KilterTransformer.git
%cd KilterTransformer
!ls

Cloning into 'KilterTransformer'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 273 (delta 55), reused 49 (delta 22), pack-reused 173 (from 1)
Receiving objects: 100% (273/273), 118.51 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (128/128), done.
Updating files: 100% (59/59), done.
/content/KilterTransformer
bert_improved.ipynb  gitignore		   main.ipynb	      req
bert.ipynb	     gpt.ipynb		   models	      saved_models
data		     gpt_shuffle.ipynb	   project_structure  src
EDA.ipynb	     gpt_shuffle_me.ipynb  pyproject.toml     utils_old
figs		     gpt_wandb.ipynb	   readme.md	      uv.lock


In [34]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorForLanguageModeling,
)
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import re
import numpy as np
from datasets import Dataset, disable_progress_bar
import wandb

disable_progress_bar()

# augment

In [35]:
def augment_route_for_set_learning(
    frames: str,
    tokenizer: PreTrainedTokenizerFast,
    num_augmentations: int = 3
    ) -> List[Dict[str, List[int]]]:
    """
    Create multiple training examples from a single route with different orderings.

    This teaches the model that holds belong together as a SET, not a SEQUENCE.

    Args:
        frames: Original route string (e.g., "angle40 grade18 start1 hand2 finish3")
        tokenizer: Tokenizer to encode the route
        num_augmentations: Number of different examples to create

    Returns:
        List of dicts with 'prefix' and 'remaining' keys
    """
    # Tokenize the full route
    tokens = tokenizer.encode(frames, add_special_tokens=True)

    # Parse structure: [BOS, angle, grade, holds..., EOS]
    bos = tokens[0]
    eos = tokens[-1]

    # Find where metadata ends and holds begin
    # Assumes format: BOS angle{N} grade{N} holds... EOS
    content_tokens = tokens[1:-1]  # Remove BOS and EOS

    # Identify metadata vs holds (simple heuristic: first 2 tokens are metadata)
    if len(content_tokens) < 2:
        # Edge case: very short sequence
        return [{
            'prefix': [bos],
            'remaining': tokens[1:]
        }]

    metadata = content_tokens[:2]  # angle and grade
    holds = content_tokens[2:]      # all holds

    augmented_examples = []

    # AUGMENTATION 1: Empty prompt (just BOS) → predict everything (5% chance)
    if np.random.random() < 0.05:
        augmented_examples.append({
            'prefix': [bos],
            'remaining': metadata + holds + [eos]
        })

    # AUGMENTATION 2: Metadata only → predict all holds + EOS (10% chance)
    if np.random.random() < 0.10:
        if len(holds) > 0:
            augmented_examples.append({
                'prefix': [bos] + metadata,
                'remaining': holds + [eos]
            })

    # AUGMENTATION 3-N: Random orderings with random split points
    for _ in range(num_augmentations):
        if len(holds) == 0:
            continue

        # Shuffle the holds
        shuffled_holds = holds.copy()
        np.random.shuffle(shuffled_holds)

        # Random split point (ensure at least 1 hold remains for prediction)
        if len(shuffled_holds) > 1:
            split_idx = np.random.randint(0, len(shuffled_holds))
        else:
            split_idx = 0

        prefix_holds = shuffled_holds[:split_idx]
        remaining_holds = shuffled_holds[split_idx:]

        augmented_examples.append({
            'prefix': [bos] + metadata + prefix_holds,
            'remaining': remaining_holds + [eos]
        })

    return augmented_examples


def create_augmented_dataset(
    dataset: Dataset,
    tokenizer: PreTrainedTokenizerFast,
    num_augmentations: int = 5
    ) -> Dataset:
    """
    Transform a dataset of routes into an augmented dataset for set-based learning.

    Each original route becomes multiple training examples with different orderings.
    """
    all_prefixes = []
    all_remaining = []

    for example in dataset:
        frames = example['frames']
        augmented = augment_route_for_set_learning(frames, tokenizer, num_augmentations)

        for aug_example in augmented:
            all_prefixes.append(aug_example['prefix'])
            all_remaining.append(aug_example['remaining'])

    # Create new dataset
    return Dataset.from_dict({
        'prefix': all_prefixes,
        'remaining': all_remaining
    })

# collator

In [36]:
@dataclass
class KilterCollator:
    """
    Collates batches for set-based prediction.

    Key difference from standard collators:
    - 'remaining' stays as List[List[int]] (not padded tensor)
    - Only 'prefix' (input_ids) is padded
    """
    tokenizer: PreTrainedTokenizerFast

    def __call__(self, examples: List[Dict]) -> Dict:
        # Extract components
        prefixes = [ex['prefix'] for ex in examples]
        remaining_sets = [ex['remaining'] for ex in examples]

        # Pad prefixes to same length
        max_prefix_len = max(len(p) for p in prefixes)

        input_ids = []
        attention_mask = []

        for prefix in prefixes:
            padding_length = max_prefix_len - len(prefix)

            # Pad on the right
            padded_ids = prefix + [self.tokenizer.pad_token_id] * padding_length
            mask = [1] * len(prefix) + [0] * padding_length

            input_ids.append(padded_ids)
            attention_mask.append(mask)

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'remaining': remaining_sets,  # Keep as list of lists!
        }


# gpt

In [37]:
class ShuffleKilterGPT(nn.Module):
    """
    GPT-2 model for set-based climbing route generation.

    Key differences from sequential model:
    1. Learns which holds belong together (set membership)
    2. Loss computed only at last position of prefix
    3. Predicts over "remaining tokens" not "next token in sequence"
    """

    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 4,
        n_layer: int = 6,
        n_positions: int = 128,
        dropout: float = 0.1
        ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        remaining: Optional[List[List[int]]] = None
        ):
        """
        Forward pass with set-based loss.

        Args:
            input_ids: [batch_size, seq_len] - prefix tokens
            attention_mask: [batch_size, seq_len] - attention mask
            remaining: List of lists - valid next tokens for each example

        Returns:
            Model outputs with custom loss
        """
        # Get base model outputs
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=None  # We compute custom loss
        )

        if remaining is not None:
            # Compute set-based loss
            logits = outputs.logits  # [batch_size, seq_len, vocab_size]
            batch_size = logits.size(0)

            # Find last non-padded position for each example
            if attention_mask is not None:
                seq_lengths = attention_mask.sum(dim=1) - 1  # [batch_size]
            else:
                seq_lengths = torch.full((batch_size,), logits.size(1) - 1, device=logits.device)

            # Extract logits at last position only
            last_logits = logits[
                torch.arange(batch_size, device=logits.device),
                seq_lengths
            ]  # [batch_size, vocab_size]

            # Compute log probabilities
            log_probs = F.log_softmax(last_logits, dim=-1)

            # Multi-label loss: log-sum-exp over valid tokens
            loss = 0.0
            for i in range(batch_size):
                if len(remaining[i]) == 0:
                    continue

                valid_tokens = torch.tensor(
                    remaining[i],
                    dtype=torch.long,
                    device=logits.device
                )

                # Get log probs for all valid tokens
                valid_log_probs = log_probs[i, valid_tokens]

                # Loss = -log P(any valid token)
                # = -log(sum_k exp(log P(token_k)))
                # = -logsumexp(log probs of valid tokens)
                loss -= torch.logsumexp(valid_log_probs, dim=0)

            outputs.loss = loss / batch_size

        return outputs

    def generate_route(
        self,
        tokenizer: PreTrainedTokenizerFast,
        angle: int = 40,
        grade: int = 18,
        max_holds: int = 20,
        temperature: float = 1.0,
        top_p: float = 0.95,
        device: str = "cpu",
        required_holds: Optional[List[str]] = None
        ) -> str:
        """
        Generate a route using iterative set-based sampling.

        Unlike standard autoregressive generation, this:
        1. Samples from distribution over remaining tokens
        2. Prevents duplicates
        3. Can enforce constraints (required holds)

        Args:
            tokenizer: Tokenizer
            angle: Board angle (20-60, multiples of 5)
            grade: Route grade (13-27)
            max_holds: Maximum number of holds to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
            device: Device to run on
            required_holds: Optional list of hold names that must be included

        Returns:
            Generated route string
        """
        self.model.eval()
        self.model.to(device)

        # Round angle to nearest 5
        angle_rounded = max(20, min(60, round(angle / 5) * 5))
        grade = max(13, min(27, grade))

        # Encode metadata
        angle_str = f"angle{angle_rounded}"
        grade_str = f"grade{grade}"

        angle_token = tokenizer.encode(angle_str, add_special_tokens=False)[0]
        grade_token = tokenizer.encode(grade_str, add_special_tokens=False)[0]

        # Start with BOS + metadata
        prefix = [
            tokenizer.bos_token_id,
            angle_token,
            grade_token
        ]

        used_tokens = set(prefix)

        # Handle required holds
        required_token_ids = []
        if required_holds:
            for hold in required_holds:
                hold_tokens = tokenizer.encode(hold, add_special_tokens=False)
                if len(hold_tokens) > 0:
                    required_token_ids.append(hold_tokens[0])

        # Generation loop
        for _ in range(max_holds):
            # Prepare input
            input_ids = torch.tensor([prefix], device=device)
            attention_mask = torch.ones_like(input_ids)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits[0, -1, :]  # Last position logits

                # Apply temperature
                logits = logits / temperature

                # Mask already-used tokens
                for token in used_tokens:
                    logits[token] = float('-inf')

                # Convert to probabilities
                probs = F.softmax(logits, dim=-1)

                # Nucleus sampling (top-p)
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumsum_probs = torch.cumsum(sorted_probs, dim=0)

                # Remove tokens with cumulative probability above threshold
                sorted_indices_to_remove = cumsum_probs > top_p
                sorted_indices_to_remove[0] = False  # Keep at least one token

                # Zero out probabilities
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                probs[indices_to_remove] = 0.0

                # Renormalize
                if probs.sum() > 0:
                    probs = probs / probs.sum()
                else:
                    # Fallback: uniform over unused tokens
                    probs = torch.ones_like(probs)
                    for token in used_tokens:
                        probs[token] = 0.0
                    probs = probs / probs.sum()

                # Sample next token
                next_token = torch.multinomial(probs, num_samples=1).item()

                # Check for EOS
                if next_token == tokenizer.eos_token_id:
                    break

                # Add to prefix
                prefix.append(next_token)
                used_tokens.add(next_token)

        # Ensure required holds are present
        if required_token_ids:
            for req_token in required_token_ids:
                if req_token not in used_tokens:
                    prefix.insert(-1, req_token)  # Insert before last token

        # Decode to string
        route_str = tokenizer.decode(prefix, skip_special_tokens=True)
        return route_str

    def validate_route(self, route_str: str) -> Tuple[bool, str]:
        """Validate that a generated route meets constraints."""
        pattern = r"angle(\d+)\s+grade(\d+)\s+(.*)"
        match = re.match(pattern, route_str)

        if not match:
            return False, "Invalid route format"

        angle, grade, holds_str = match.groups()
        angle, grade = int(angle), int(grade)

        if not (20 <= angle <= 60):
            return False, f"Invalid angle: {angle}"
        if not (13 <= grade <= 27):
            return False, f"Invalid grade: {grade}"

        holds = holds_str.split()
        num_start = sum(1 for h in holds if h.startswith("start"))
        num_finish = sum(1 for h in holds if h.startswith("finish"))

        if not (1 <= num_start <= 2):
            return False, f"Must have 1-2 start holds, got {num_start}"
        if not (1 <= num_finish <= 2):
            return False, f"Must have 1-2 finish holds, got {num_finish}"
        if len(holds) >= 25:
            return False, f"Too many holds: {len(holds)}"

        return True, "Valid"


# trainer

In [38]:
class KilterTrainer(Trainer):
    """
    Custom Trainer that handles:
    - Training: Custom prefix/remaining format with set-based loss
    - Evaluation: Standard input_ids format with standard GPT loss
    """

    def __init__(self, eval_collator=None, **kwargs):
        super().__init__(**kwargs)
        self.eval_collator = eval_collator  # Separate collator for eval

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Custom loss computation that handles 'remaining' parameter for training.
        """
        # Check if this is custom format (has 'remaining') or standard format
        if "remaining" in inputs:
            # Training with custom format
            remaining = inputs.pop("remaining", None)

            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                remaining=remaining
            )

            if remaining is not None:
                inputs["remaining"] = remaining

            loss = outputs.loss
        else:
            # Evaluation with standard format - use standard GPT loss
            labels = inputs.pop("labels", None)

            # Call the inner GPT2 model directly for standard loss
            outputs = model.model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                labels=labels
            )

            if labels is not None:
                inputs["labels"] = labels

            loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """
        Override prediction step for evaluation.
        Handles both custom format (training) and standard format (eval).
        """
        # Check format
        if "remaining" in inputs:
            # Custom format
            remaining = inputs.pop("remaining", None)
            inputs = self._prepare_inputs(inputs)

            with torch.no_grad():
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask"),
                    remaining=remaining
                )
                loss = outputs.loss
                logits = outputs.logits
        else:
            # Standard format - use standard GPT evaluation
            labels = inputs.get("labels")
            inputs = self._prepare_inputs(inputs)

            with torch.no_grad():
                outputs = model.model(  # Use inner GPT2 model
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask"),
                    labels=inputs.get("labels")
                )
                loss = outputs.loss if labels is not None else None
                logits = outputs.logits

        if prediction_loss_only:
            return (loss, None, None)

        return (loss, logits, inputs.get("labels"))

    def get_eval_dataloader(self, eval_dataset=None):
        """
        Override to use eval_collator for evaluation if provided.
        """
        if eval_dataset is None:
            eval_dataset = self.eval_dataset

        # Use eval_collator if provided, otherwise use train collator
        data_collator = self.eval_collator if self.eval_collator is not None else self.data_collator

        eval_sampler = self._get_eval_sampler(eval_dataset)

        return torch.utils.data.DataLoader(
            eval_dataset,
            sampler=eval_sampler,
            batch_size=self.args.eval_batch_size,
            collate_fn=data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def get_test_dataloader(self, test_dataset):
        """
        Override to use eval_collator for test if provided.
        """
        data_collator = self.eval_collator if self.eval_collator is not None else self.data_collator

        test_sampler = self._get_eval_sampler(test_dataset)

        return torch.utils.data.DataLoader(
            test_dataset,
            sampler=test_sampler,
            batch_size=self.args.eval_batch_size,
            collate_fn=data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def _save(self, output_dir: Optional[str] = None, state_dict=None):
        """
        Override save to handle the ShuffleKilterGPT wrapper.
        Save the inner GPT2 model, not the wrapper.
        """
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        # Save the inner GPT2 model (which handles shared weights properly)
        if hasattr(self.model, 'model'):
            self.model.model.save_pretrained(
                output_dir,
                state_dict=state_dict,
                safe_serialization=True
            )
        else:
            # Fallback to standard save
            super()._save(output_dir, state_dict)


# helper

In [39]:
def load_model(model_path: str, device: str = "cpu") -> ShuffleKilterGPT:
    """Load a trained set-based model."""
    gpt2_model = GPT2LMHeadModel.from_pretrained(model_path)
    model = ShuffleKilterGPT(vocab_size=gpt2_model.config.vocab_size)
    model.model = gpt2_model
    model.config = gpt2_model.config
    model.to(device)
    model.eval()
    return model


def preprocess_datasets_for_set_learning(
    datasets: Dict[str, Dataset],
    tokenizer: PreTrainedTokenizerFast,
    num_augmentations: int = 3
    ) -> Dict[str, Dataset]:
    """
    Preprocess datasets for set-based learning.

    - TRAIN: Shuffled augmentation (teaches set membership, order-invariance)
           Uses custom prefix/remaining format
    - VAL/TEST: Standard tokenization (no augmentation, standard GPT evaluation)
           Uses standard input_ids format for DataCollatorForLanguageModeling
    """
    processed_datasets = {}

    # Train: Augmented with shuffling (custom format)
    print(f"Augmenting train split with shuffling...")
    original_size = len(datasets['train'])
    processed_datasets['train'] = create_augmented_dataset(
        datasets['train'],
        tokenizer,
        num_augmentations=num_augmentations
    )
    augmented_size = len(processed_datasets['train'])
    print(f"  {original_size} routes → {augmented_size} training examples (shuffled)")

    # Val: Standard tokenization (no custom format)
    print(f"Tokenizing val split (standard format)...")
    original_size = len(datasets['val'])
    processed_datasets['val'] = datasets['val'].map(
        lambda example: tokenizer(example["frames"]),
        batched=True,
        remove_columns=datasets['val'].column_names
    )
    print(f"  {original_size} routes (standard GPT evaluation)")

    # Test: Standard tokenization (no custom format)
    print(f"Tokenizing test split (standard format)...")
    original_size = len(datasets['test'])
    processed_datasets['test'] = datasets['test'].map(
        lambda example: tokenizer(example["frames"]),
        batched=True,
        remove_columns=datasets['test'].column_names
    )
    print(f"  {original_size} routes (standard GPT evaluation)")

    return processed_datasets

# main

In [40]:
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer

# Initialize wandb
wandb.init(
    project="climb-gpt-shuffle",
    name=f"gpt_shuffle_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config={
        "architecture": "SetBasedGPT",
        "n_embd": 256,
        "n_head": 4,
        "n_layer": 6,
        "n_positions": 128,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": 16,
        "learning_rate": 1e-5,
        "weight_decay": 0.01,
        "gradient_accumulation_steps": 1,
        "early_stopping_patience": 5,
        "augmentation_factor": 5,
    }
)

run_name = wandb.run.name
OUT_DIR = f"/content/drive/MyDrive/KilterTransformer/models/climb_gpt_shuffle/{run_name}"
# OUT_DIR = f"models/climb_gpt_shuffle/{run_name}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load data
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)
dp = DataPreprocessing()
datasets = dp.load_climbs()

# Split data
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

print(f"Original dataset sizes:")
print(f"  Train: {len(datasets['train'])}")
print(f"  Val:   {len(datasets['val'])}")
print(f"  Test:  {len(datasets['test'])}")

# Train tokenizer
print("\n" + "="*60)
print("TRAINING TOKENIZER")
print("="*60)
tokenizer = train_tokenizer(datasets, OUT_DIR)
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Augment datasets for set-based learning
print("\n" + "="*60)
print("AUGMENTING DATA FOR SET-BASED LEARNING")
print("="*60)
datasets = preprocess_datasets_for_set_learning(
    datasets,
    tokenizer,
    num_augmentations=3
)

# Log dataset info
wandb.config.update({
    "train_size_augmented": len(datasets['train']),
    "val_size_augmented": len(datasets['val']),
    "test_size_augmented": len(datasets['test']),
    "vocab_size": tokenizer.vocab_size,
})

# Create model
print("\n" + "="*60)
print("INITIALIZING MODEL")
print("="*60)
model = ShuffleKilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Watch model with wandb
wandb.watch(model.model, log="all", log_freq=1000)

# Create data collators
# Train collator: Custom format (prefix/remaining)
train_collator = KilterCollator(tokenizer=tokenizer)

# Eval collator: Standard GPT format (input_ids/labels)
eval_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


train/epoch,▁▃▅▆█
train/global_step,▁▃▅▆█
train/grad_norm,█▇▆▄▁
train/learning_rate,█▆▅▃▁
train/loss,█▅▃▂▁
train/epoch,0.04123
train/global_step,500
train/grad_norm,9.42203
train/learning_rate,1e-05
train/loss,1.0322


Using device: cuda

LOADING DATA
Loaded 76992 routes from cache data/climbs_cleaned.csv
Original dataset sizes:
  Train: 61593
  Val:   7699
  Test:  7700

TRAINING TOKENIZER
Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('start1345', 1117), ('feet1471', 1416), ('finish1486', 1479), ('finish1523', 1627), ('hand1394', 1314), ('feet1297', 924), ('start1129', 253), ('start1376', 1241), ('hand1494', 1510), ('finish1090', 99)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to /content/drive/MyDrive/KilterTransformer/models/climb_gpt_shuffle/gpt_shuffle_20251102_181021
Vocabulary size: 1932

AUGMENTING DATA FOR SET-BASED LEARNING
Augmenting train split with s

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,
    eval_steps=500, ##############
    save_steps=500, ##############
    num_train_epochs=30, ##############
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    run_name=run_name,
)

# Create trainer with custom trainer class
trainer = KilterTrainer(
    model=model,
    args=training_args,
    data_collator=train_collator,  # For training (custom format)
    eval_collator=eval_collator,    # For evaluation (standard format)
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)


trainer.train()

model.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"✓ Model saved to {OUT_DIR}")

# Log model as artifact
artifact = wandb.Artifact(
    name=f"climb-gpt-set-model-{run_name}",
    type="model",
    description="Set-based KilterGPT model"
)
artifact.add_dir(OUT_DIR)
wandb.log_artifact(artifact)

test_results = trainer.evaluate(datasets["test"])
print(f"✓ Test Loss: {test_results['eval_loss']:.4f}")

# Log test results
wandb.log({
    "test/loss": test_results['eval_loss'],
})

wandb.summary["final_test_loss"] = test_results['eval_loss']
wandb.summary["model_path"] = OUT_DIR



Step,Training Loss,Validation Loss


In [ ]:
# Generate sample routes
print("\n" + "="*60)
print("GENERATING SAMPLE ROUTES")
print("="*60)

model.to(device)

test_configs = [
    {"angle": 40, "grade": 18},
    {"angle": 30, "grade": 15},
    {"angle": 50, "grade": 22},
]

for config in test_configs:
    print(f"\nGenerating route: angle={config['angle']}, grade={config['grade']}")
    for i in range(3):
        route = model.generate_route(
            tokenizer=tokenizer,
            angle=config['angle'],
            grade=config['grade'],
            temperature=0.9,
            device=device
        )
        is_valid, msg = model.validate_route(route)
        print(f"  {i+1}. {route}")
        print(f"     Valid: {is_valid} {f'({msg})' if not is_valid else ''}")

wandb.finish()
print(f"\n✓ Training complete! View results at: {wandb.run.url}")